# 03 — Différencier les RÔLES : soliste / accompagnement / dunun

Chaque **piste** a un rôle connu : jembé **soliste** (J1), jembé **accompagnement** (J2), **dunun** (D1/D2). On apprend à le retrouver à partir de 4 indices :
- **densité** de frappes (frappes/s),
- **déterminisme** du motif (0 = varié → soliste ; 1 = cyclique → accompagnement),
- **centroïde** spectral et **ratio grave** (le dunun est grave).

Tâche **supervisée** (labels connus), évaluée en **leave-one-out**.

## 0. Imports & table de features par piste (~1 min)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix
import djembe_utils as dj

c = dj.list_corpus(); rows = []
for _, r in c.iterrows():
    on = dj.load_onsets(r['piece'], r['take'], r['code'])
    if on is None or len(on) < 5:
        continue
    cen, low = dj.track_timbre(dj.load_track(r['piece'], r['take'], r['code']))
    span = on['Time'].max() - on['Time'].min()
    rows.append({'piece':r['piece'], 'take':r['take'], 'code':r['code'], 'role':r['role'],
                 'density':len(on)/span, 'determ':dj.determinism(r['piece'], r['take'], r['code']),
                 'centroid':cen, 'low_ratio':low})
roles = pd.DataFrame(rows).dropna()
print(len(roles), 'pistes'); display(roles.groupby('role')[['density','determ','centroid','low_ratio']].mean().round(2))

## 1. Voir la séparation
Déterminisme (axe X) vs centroïde (axe Y) suffisent presque : le dunun est en bas (grave), l'accompagnement à droite (cyclique), le soliste à gauche (varié).

In [ ]:
COL = {'jembe_solo':'#d63bd6', 'jembe_accomp':'#9b59b6', 'dunun':'#2b6cff'}
plt.figure(figsize=(8,6))
for role, cc in COL.items():
    m = roles['role']==role
    plt.scatter(roles.loc[m,'determ'], roles.loc[m,'centroid'], c=cc, label=role, s=70, alpha=.8)
plt.xlabel('déterminisme du motif (0=varié, 1=cyclique)'); plt.ylabel('centroïde spectral (Hz)')
plt.legend(); plt.title('Pistes par rôle'); plt.grid(alpha=.2); plt.show()

## 2. Classifieur (leave-one-out)

In [ ]:
F = ['density', 'determ', 'centroid', 'low_ratio']
pred = cross_val_predict(RandomForestClassifier(n_estimators=300, random_state=0),
                         roles[F].values, roles['role'].values, cv=LeaveOneOut())
print('Accuracy leave-one-out :', round(accuracy_score(roles['role'], pred), 3))
labs = ['jembe_solo', 'jembe_accomp', 'dunun']
display(pd.DataFrame(confusion_matrix(roles['role'], pred, labels=labs),
                     index=['vrai '+l for l in labs], columns=['préd '+l for l in labs]))

## Bilan
- Les 3 rôles se séparent très bien (le **déterminisme** distingue soliste vs accompagnement, le **timbre** isole le dunun).
- C'est la brique pour, plus tard, **annoter automatiquement** qui joue quoi dans un enregistrement.
- Suite : `04_nombre` (compter les percussions) et `05_rythme` (reconnaître la pièce).